# LocPop: reproducible clustering benchmark

Shared experiment logic can be found in
`BenchmarkNumbaExperiments.py`, which prevents the LocPop and LocStab protocols
from drifting apart.

The run uses fixed data seeds, ten paired permutations, standardized vector
features, disjoint friendship/enmity relations, correctly named adjusted Rand
index (ARI), same-run warm starts, convergence diagnostics, and explicit
algorithm parameters.  It writes both raw and summarized CSVs and regenerates
individual plus four-panel summary plots.

Each clustering dataset is evaluated from singleton (S), predicted-$k$ (P),
$k$-means (KM), and DBSCAN (D) initializations.  The coalition-label capacity
is $n$, so every non-singleton partition permits a singleton-creation move
through an empty label.

Expected outputs:

- `csv/PopularClustering/runs.csv`
- `csv/PopularClustering/results.csv`
- `csv/PopularClustering/preprocessing.csv`
- `csv/PopularClustering/dataset-0.csv`, `dataset-1.csv`, `dataset-2.csv`
- `figures/PopularClustering/*.png`


In [1]:
from importlib.metadata import version
from BenchmarkNumbaExperiments import run_clustering_experiment, DATA_SEED, THRESHOLDS, DOMAINS

REPETITIONS = 10
LOCAL_STABLE = False

print("Data seed:", DATA_SEED)
print("Repetitions:", REPETITIONS)
print("Thresholds:", THRESHOLDS)
print("Domains:", DOMAINS)

for package in ["numpy", "scipy", "pandas", "scikit-learn", "networkx", "numba", "matplotlib"]:
    print(f"{package}={version(package)}")


Data seed: 20260817
Repetitions: 10
Thresholds: ((0.2, 0.2), (0.25, 0.35), (0.4, 0.4))
Domains: ('B', 'AF', 'AE')
numpy=2.5.2
scipy=1.18.0
pandas=3.0.5
scikit-learn=1.9.0
networkx=3.6.1
numba=0.67.0
matplotlib=3.11.1


## Execute the complete experiment

This cell performs the full production run and overwrites the corresponding
CSV and figure artifacts.  Relationship preprocessing and the complete grid of
initializations can take some time.  Do not interrupt the
kernel while files are being written.


In [2]:
summary = run_clustering_experiment(local_stable=LOCAL_STABLE, repetitions=REPETITIONS)
summary

,Method,Dataset,Preference,Initialization,Beta Friend,Beta Enemy,Repetitions,Adjusted Rand Index,Adjusted Rand Index SD,Silhouette Score,...,Seconds,Seconds SD,Moves,Moves SD,Converged,Converged SD,Final Coalitions,Final Coalitions SD,Initial Adjusted Rand Index,Initial Adjusted Rand Index SD
0,KMeans,Moons,-,-,NaN,NaN,10,0.478969,5.551115e-17,0.487960,...,0.306662,0.659702,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,DBSCAN,Moons,-,-,NaN,NaN,10,0.993355,0.000000e+00,0.017502,...,0.004284,0.001362,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,LocPop,Moons,B,S,0.2,0.2,10,0.227709,7.743995e-03,0.464808,...,0.130840,0.146773,364.8,9.239048,1.0,0.0,10.2,0.4,0.000000,0.000000
3,LocPop,Moons,B,P,0.2,0.2,10,0.230663,5.921500e-03,0.438136,...,0.039770,0.002876,303.3,15.231874,1.0,0.0,10.6,0.8,0.001426,0.002191
4,LocPop,Moons,B,KM,0.2,0.2,10,0.235948,2.053993e-04,0.465869,...,0.032977,0.003127,237.2,4.237924,1.0,0.0,10.0,0.0,0.187561,0.000943
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
147,LocPop,Iris,AF,D,0.4,0.4,10,0.550942,1.110223e-16,0.426657,...,0.005038,0.000649,53.0,0.000000,1.0,0.0,4.0,0.0,0.000000,0.000000
148,LocPop,Iris,AE,S,0.4,0.4,10,0.550942,1.110223e-16,0.426657,...,0.010396,0.001269,149.0,3.000000,1.0,0.0,4.0,0.0,0.000000,0.000000
149,LocPop,Iris,AE,P,0.4,0.4,10,0.550942,1.110223e-16,0.426657,...,0.006806,0.000787,117.1,5.262129,1.0,0.0,4.0,0.0,-0.004360,0.001956
150,LocPop,Iris,AE,KM,0.4,0.4,10,0.550942,1.110223e-16,0.426657,...,0.004898,0.000778,46.7,0.900000,1.0,0.0,4.0,0.0,0.554128,0.001718


## Verify convergence and output coverage

In [3]:
heuristic = summary[summary["Method"] == "LocPop"]
print("Summary rows:", len(summary))
print("Datasets:", sorted(summary["Dataset"].unique()))
print("Minimum convergence rate:", heuristic["Converged"].min())
print("Maximum recorded moves:", heuristic["Moves"].max())

if not (heuristic["Converged"] == 1.0).all():
    display(heuristic[heuristic["Converged"] < 1.0])
    raise RuntimeError("At least one run reached the move cap; do not use the outputs without investigation.")

display(summary.sort_values(["Dataset", "Method", "Initialization", "Preference"]).head(20))
print("Artifacts written under csv/PopularClustering and figures/PopularClustering")


Summary rows: 152
Datasets: ['3 Circles', 'Cancer', 'Iris', 'Moons']
Minimum convergence rate: 1.0
Maximum recorded moves: 586.0


,Method,Dataset,Preference,Initialization,Beta Friend,Beta Enemy,Repetitions,Adjusted Rand Index,Adjusted Rand Index SD,Silhouette Score,...,Seconds,Seconds SD,Moves,Moves SD,Converged,Converged SD,Final Coalitions,Final Coalitions SD,Initial Adjusted Rand Index,Initial Adjusted Rand Index SD
39,DBSCAN,3 Circles,-,-,NaN,NaN,10,0.187167,4.282094e-04,0.126546,...,0.003914,0.000707,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
38,KMeans,3 Circles,-,-,NaN,NaN,10,0.444278,1.412614e-03,0.409868,...,0.093981,0.024626,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
51,LocPop,3 Circles,AE,D,0.20,0.20,10,0.284361,3.189539e-03,0.263374,...,0.027924,0.002598,160.3,2.570992,1.0,0.0,9.0,0.000000,0.344372,0.003307
63,LocPop,3 Circles,AE,D,0.25,0.35,10,0.425512,5.551115e-17,0.288695,...,0.026846,0.002588,160.2,1.469694,1.0,0.0,4.0,0.000000,0.273134,0.000382
75,LocPop,3 Circles,AE,D,0.40,0.40,10,0.362144,0.000000e+00,0.456650,...,0.036323,0.003825,242.4,0.489898,1.0,0.0,2.0,0.000000,0.184120,0.000204
47,LocPop,3 Circles,AF,D,0.20,0.20,10,0.289859,0.000000e+00,0.307381,...,0.028863,0.007087,161.2,1.777639,1.0,0.0,8.0,0.000000,0.344713,0.000983
59,LocPop,3 Circles,AF,D,0.25,0.35,10,0.417107,0.000000e+00,0.288812,...,0.024649,0.001928,158.0,1.095445,1.0,0.0,4.0,0.000000,0.279407,0.000380
71,LocPop,3 Circles,AF,D,0.40,0.40,10,0.365679,5.551115e-17,0.456512,...,0.038676,0.004526,242.4,0.489898,1.0,0.0,2.0,0.000000,0.188099,0.000202
43,LocPop,3 Circles,B,D,0.20,0.20,10,0.286226,3.720179e-03,0.343830,...,0.028390,0.006842,158.6,2.009975,1.0,0.0,7.6,0.489898,0.342355,0.005190
55,LocPop,3 Circles,B,D,0.25,0.35,10,0.417107,0.000000e+00,0.288812,...,0.029333,0.007320,158.0,1.095445,1.0,0.0,4.0,0.000000,0.279407,0.000380


Artifacts written under csv/PopularClustering and figures/PopularClustering
